Primo test - RNN con Attention

1. Preparazione ambiente (importazione librerie e controllo GPU)

In [1]:
import pandas as pd
import numpy as np
import re
import torch

print(f"PyTorch Version: {torch.__version__}")
# Impostiamo la GPU come device principale (se disponibile)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sto usando: {device}")

PyTorch Version: 2.7.1+cu118
Sto usando: cuda


2. Analisi e preprocessing dei dati (caricamento del file berg.txt e pulizia del testo)

In [2]:
def preprocess_sentence(w):
    # Tutto in minuscolo e togliamo gli spazi all'inizio e alla fine
    w = str(w).lower().strip()

    # Stacchiamo la punteggiatura dalle parole per considerarla come token separati
    # Es: "ciao." diventa "ciao ."
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)

    # Teniamo le lettere base, i numeri, la punteggiatura e TUTTI gli accenti tipici dei dialetti/italiano
    # Includiamo à, è, é, ì, í, ò, ó, ù, ú, ö, ü
    w = re.sub(r"[^a-zàèéìíòóùúöü0-9?.!,¿']+", " ", w)
    
    w = w.strip()
    
    # Aggiungiamo i token speciali per far capire alla rete quando inizia e finisce la frase
    w = '<sos> ' + w + ' <eos>'
    return w

# Facciamo un test veloce per vedere se funziona
test_phrase = "L'è svèelt com ü fölmen." # Dal tuo file berg.txt
print("Prima:", test_phrase)
print("Dopo: ", preprocess_sentence(test_phrase))

Prima: L'è svèelt com ü fölmen.
Dopo:  <sos> l'è svèelt com ü fölmen . <eos>


3. Architettura del vocabolario (tokenizzazione e indicizzazione)

In [3]:
# Leggiamo il file ignorando la terza colonna (le fonti)
df = pd.read_csv("berg.txt", sep="\t", header=None, usecols=[0, 1], names=["Bergamasco", "Italiano"])

# Puliamo eventuali righe vuote
df = df.dropna()

# Applichiamo il preprocessing a entrambe le colonne
df['Bergamasco_pulito'] = df['Bergamasco'].apply(preprocess_sentence)
df['Italiano_pulito'] = df['Italiano'].apply(preprocess_sentence)

# Mostriamo le prime 5 righe
display(df.head())

,Bergamasco,Italiano,Bergamasco_pulito,Italiano_pulito
0,Visiù,Visione,<sos> visiù <eos>,<sos> visione <eos>
1,In mès a sbarlafüs e mes-ciotade,In mezzo ad oggetti e cianfrusaglie,<sos> in mès a sbarlafüs e mes ciotade <eos>,<sos> in mezzo ad oggetti e cianfrusaglie <eos>
2,ó troàt la sò curuna del rosare...,ho trovato la sua corona del rosario...,<sos> ó troàt la sò curuna del rosare . . . <eos>,<sos> ho trovato la sua corona del rosario . ....
3,la giràa söi nodèi a belasi,la girava piano piano sopra le nocchne,<sos> la giràa söi nodèi a belasi <eos>,<sos> la girava piano piano sopra le nocchne <...
4,insèma l'mururà,insieme al mormorare,<sos> insèma l'mururà <eos>,<sos> insieme al mormorare <eos>


In [4]:
class Vocabulary:
    def __init__(self):
        # Inizializziamo i dizionari con i 4 token speciali di base
        self.word2index = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.index2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.n_words = 4  # Contatore delle parole (partiamo da 4 perché abbiamo già i token speciali)

    def add_sentence(self, sentence):
        # Dividiamo la frase in singole parole (sugli spazi)
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        # Se la parola non è mai stata vista, la aggiungiamo e le diamo un numero (ID)
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

# Creiamo le due "scatole" vuote per le due lingue
vocab_berg = Vocabulary()
vocab_ita = Vocabulary()

# Leggiamo riga per riga dal tuo file excel (DataFrame) e popoliamo i vocabolari
for frase in df['Bergamasco_pulito']:
    vocab_berg.add_sentence(frase)

for frase in df['Italiano_pulito']:
    vocab_ita.add_sentence(frase)

print(f"Grandezza vocabolario Bergamasco: {vocab_berg.n_words} parole uniche")
print(f"Grandezza vocabolario Italiano: {vocab_ita.n_words} parole uniche")

# Facciamo un test divertente: vediamo che numero ha assegnato alla parola "visiù" (se esiste nel tuo file)
parola_test = "visiù"
if parola_test in vocab_berg.word2index:
    print(f"L'ID della parola '{parola_test}' è: {vocab_berg.word2index[parola_test]}")

Grandezza vocabolario Bergamasco: 424 parole uniche
Grandezza vocabolario Italiano: 454 parole uniche
L'ID della parola 'visiù' è: 4


4. Pipeline di input per PyTorch (dataset e DataLoader)

In [5]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# 1. Definiamo come PyTorch deve leggere e trasformare le frasi
class DialectDataset(Dataset):
    def __init__(self, df, vocab_berg, vocab_ita):
        self.df = df
        self.vocab_berg = vocab_berg
        self.vocab_ita = vocab_ita

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Prendiamo le frasi pulite
        frase_berg = self.df.iloc[idx]['Bergamasco_pulito']
        frase_ita = self.df.iloc[idx]['Italiano_pulito']

        # Convertiamo ogni parola nel suo numero ID. Se una parola non c'è, mettiamo 3 (<unk>)
        tensor_berg = [self.vocab_berg.word2index.get(w, 3) for w in frase_berg.split(' ')]
        tensor_ita = [self.vocab_ita.word2index.get(w, 3) for w in frase_ita.split(' ')]

        # Restituiamo i due tensori di numeri
        return torch.tensor(tensor_ita), torch.tensor(tensor_berg)

# 2. Definiamo la funzione per "pareggiare" le lunghezze con gli zeri
def pad_collate(batch):
    berg_batch, ita_batch = zip(*batch)
    
    # Mettiamo gli zeri (padding_value=0) alla fine delle sequenze più corte
    berg_padded = pad_sequence(berg_batch, batch_first=True, padding_value=0)
    ita_padded = pad_sequence(ita_batch, batch_first=True, padding_value=0)
    
    return berg_padded, ita_padded

# 3. Creiamo l'oggetto Dataset
dataset = DialectDataset(df, vocab_berg, vocab_ita)

# 4. Creiamo il DataLoader (il "cameriere" che porta i dati alla GPU). 
# Impostiamo batch_size=2 giusto per testare sul tuo file piccolino.
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=pad_collate)

# Facciamo un test: chiediamo al DataLoader di portarci il primissimo batch
for berg_b, ita_b in dataloader:
    print("BATCH BERGAMASCO (Tensore di Numeri):")
    print(berg_b)
    print(f"Dimensioni: {berg_b.shape} (2 frasi, lunghezza massima {berg_b.shape[1]})")
    
    print("\nBATCH ITALIANO (Tensore di Numeri):")
    print(ita_b)
    print(f"Dimensioni: {ita_b.shape} (2 frasi, lunghezza massima {ita_b.shape[1]})")
    break # Usiamo break per fermarci dopo aver visto solo il primo batch

BATCH BERGAMASCO (Tensore di Numeri):
tensor([[  1, 283, 284,  51, 285, 286, 287,  18,   2],
        [  1, 414,  84, 250,  18,   2,   0,   0,   0]])
Dimensioni: torch.Size([2, 9]) (2 frasi, lunghezza massima 9)

BATCH ITALIANO (Tensore di Numeri):
tensor([[  1, 175, 275,  90, 276, 277, 278,  19,   2],
        [  1, 165, 393, 154, 244,  19,   2,   0,   0]])
Dimensioni: torch.Size([2, 9]) (2 frasi, lunghezza massima 9)


5. Definizione del modello (RNN con attention)

In [6]:
import torch.nn as nn

class RnnEncoder(nn.Module):
    def __init__(self, src_vocab, embedding_dim, hidden_units):
        super(RnnEncoder, self).__init__()
        # Usiamo .n_words come definito nella nostra classe Vocabulary
        self.src_vocab = src_vocab 
        vocab_size = src_vocab.n_words 

        # Inizializza il layer di Embedding
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)

        # Inizializza la GRU (batch_first=False come voleva il prof)
        self.single_gru = nn.GRU(input_size=embedding_dim, hidden_size=hidden_units, num_layers=1, batch_first=False)

    def forward(self, x):
        # Siccome il nostro dataloader crea tensori [batch_size, max_len], 
        # li giriamo (trasponiamo) in [max_len, batch_size] per far felice la GRU del prof.
        x = x.transpose(0, 1) 

        # Passiamo x nell'embedding e poi nella rete ricorrente
        embedded = self.embedding_layer(x)
        output, hidden_state = self.single_gru(embedded)

        return output, hidden_state

In [7]:
import torch.nn.functional as F

class RnnDecoder(nn.Module):
    def __init__(self, trg_vocab, embedding_dim, hidden_units):
        super(RnnDecoder, self).__init__()
        # Adattamento per la nostra classe Vocabulary
        self.trg_vocab = trg_vocab
        vocab_size = trg_vocab.n_words

        # Initialize embedding layer
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)

        # Initialize layers to compute attention score
        self.w_1 = nn.Linear(hidden_units, hidden_units)
        self.w_2 = nn.Linear(hidden_units, hidden_units)
        self.v_a = nn.Linear(hidden_units, 1)

        # GRU Decoder
        self.single_gru = nn.GRU(input_size=embedding_dim+hidden_units, hidden_size=hidden_units, num_layers=1, batch_first=True)

        # Output layer
        self.fully_connected_layer = nn.Linear(hidden_units, vocab_size)


    def compute_attention(self, dec_hs, enc_output):
        dec_hs_perm = dec_hs.permute(1,0,2)
        enc_output_perm = enc_output.permute(1,0,2)
        score = self.v_a(torch.tanh(self.w_1(enc_output_perm) + self.w_2(dec_hs_perm)))

        attention_weights = F.softmax(score, dim=1)
        context_vector = torch.sum(attention_weights*enc_output_perm, dim=1)

        return context_vector, attention_weights

    def forward(self, x, dec_hs, enc_output):
        context_vector, attention_weights = self.compute_attention(dec_hs, enc_output)
        embedding_vectors = self.embedding_layer(x)
        concatenate = torch.cat((embedding_vectors, context_vector.unsqueeze(1)), dim=2)
        
        output, dec_hs = self.single_gru(concatenate, dec_hs)
        fc_out = self.fully_connected_layer(output.squeeze(1))

        return fc_out, dec_hs, attention_weights

    def decode_step(self, inputs, enc_output, dec_hs):
        assert inputs.shape[0] == enc_output.shape[1] == dec_hs.shape[1], 'batch_size must be the same across tensors'
        fc_out, dec_hs, attention_weights = self(inputs[:,-1].unsqueeze(1), dec_hs, enc_output)
        return fc_out, dec_hs

6. Training e evoluzione della loss

In [8]:
import torch.optim as optim
import time

# Questa funzione serve per dire alla rete quanto sta sbagliando. 
# Ignoriamo il token <pad> (ID 0) così non viene penalizzata per gli zeri finali.
def loss_function(real, pred):
    mask = real.ge(1).float() 
    loss_ = F.cross_entropy(pred, real, reduction='none') * mask
    return torch.mean(loss_)

def train_translation_model(encoder, decoder, dataloader, vocab_ita, epochs=10):
    # Gli Ottimizzatori sono i meccanismi che aggiornano i pesi (i "neuroni") per farli migliorare
    encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001)
    
    print("Inizio Addestramento...")
    
    for epoch in range(epochs):
        start_time = time.time()
        total_loss = 0
        
        encoder.train()
        decoder.train()
        
        # Prendiamo un blocco di frasi alla volta (un Batch)
        for batch_idx, (src, trg) in enumerate(dataloader):
            # Spostiamo i dati sulla Scheda Video
            src, trg = src.to(device), trg.to(device)
            batch_size = src.size(0)
            
            encoder_optimizer.zero_grad()
            decoder_optimizer.zero_grad()
            loss = 0
            
            # 1. L'ENCODER LEGGE IL BERGAMASCO: gli diamo le frasi e lui ci restituisce il "Riassunto Mentale"
            enc_output, dec_hidden = encoder(src)
            
            # 2. INIZIAMO A GENERARE L'ITALIANO: Inizializziamo il Decoder dicendogli "Inizia la frase" (<sos>, che per noi ha ID 1)
            dec_input = torch.tensor([[1]] * batch_size, device=device) 
            
            # 3. CICLO DEL TEMPO: Facciamo generare al Decoder una parola alla volta
            for t in range(1, trg.size(1)):
                # Il decoder "spara" le sue previsioni guardando la parola precedente e il riassunto dell'Encoder
                predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
                
                # Calcoliamo quanto la parola prevista sia lontana da quella reale che doveva indovinare
                loss += loss_function(trg[:, t], predictions)
                
                # TEACHER FORCING: Invece di dargli in pasto la sua previsione (magari sbagliata), 
                # lo "forziamo" dandogli in pasto la parola VERA come prossimo input, per aiutarlo a imparare più in fretta.
                dec_input = trg[:, t].unsqueeze(1)
                
            batch_loss = loss / int(trg.size(1))
            total_loss += batch_loss.item()
            
            # AGGIORNAMENTO DEI NEURONI: Questa è la vera e propria "apprendimento" (Backpropagation)
            batch_loss.backward()
            encoder_optimizer.step()
            decoder_optimizer.step()
            
        print(f'Epoca {epoch+1:02d}/{epochs} | Errore (Loss): {total_loss/len(dataloader):.4f} | Tempo: {time.time()-start_time:.1f}s')

# ==========================================
# AVVIO DEL MOTORE (ITALIANO -> BERGAMASCO)
# ==========================================
EMBEDDING_DIM = 256
HIDDEN_UNITS = 256

# L'Encoder ora usa il vocabolario Italiano per leggere
encoder_model = RnnEncoder(vocab_ita, EMBEDDING_DIM, HIDDEN_UNITS).to(device)

# Il Decoder ora usa il vocabolario Bergamasco per generare
decoder_model = RnnDecoder(vocab_berg, EMBEDDING_DIM, HIDDEN_UNITS).to(device)

# Avviamo l'addestramento dicendo alla funzione che il target è vocab_berg
train_translation_model(encoder_model, decoder_model, dataloader, vocab_berg, epochs=50)

Inizio Addestramento...
Epoca 01/50 | Errore (Loss): 3.7858 | Tempo: 8.3s
Epoca 02/50 | Errore (Loss): 2.9117 | Tempo: 4.5s
Epoca 03/50 | Errore (Loss): 2.2462 | Tempo: 3.6s
Epoca 04/50 | Errore (Loss): 1.6104 | Tempo: 3.7s
Epoca 05/50 | Errore (Loss): 1.0406 | Tempo: 3.3s
Epoca 06/50 | Errore (Loss): 0.6685 | Tempo: 8.4s
Epoca 07/50 | Errore (Loss): 0.4040 | Tempo: 4.9s
Epoca 08/50 | Errore (Loss): 0.2384 | Tempo: 4.0s
Epoca 09/50 | Errore (Loss): 0.1393 | Tempo: 4.0s
Epoca 10/50 | Errore (Loss): 0.0912 | Tempo: 2.0s
Epoca 11/50 | Errore (Loss): 0.0637 | Tempo: 2.0s
Epoca 12/50 | Errore (Loss): 0.0463 | Tempo: 2.1s
Epoca 13/50 | Errore (Loss): 0.0349 | Tempo: 1.9s
Epoca 14/50 | Errore (Loss): 0.0284 | Tempo: 2.2s
Epoca 15/50 | Errore (Loss): 0.0239 | Tempo: 7.3s
Epoca 16/50 | Errore (Loss): 0.0206 | Tempo: 3.7s
Epoca 17/50 | Errore (Loss): 0.0180 | Tempo: 3.7s
Epoca 18/50 | Errore (Loss): 0.0155 | Tempo: 3.8s
Epoca 19/50 | Errore (Loss): 0.0137 | Tempo: 3.4s
Epoca 20/50 | Errore (Loss

7. Test di inferenza e risultati

In [20]:
def translate_ita_to_berg(sentence, encoder, decoder, vocab_src, vocab_trg, max_length=50):
    encoder.eval()
    decoder.eval()

    sentence_pulita = preprocess_sentence(sentence)
    
    # Usiamo il vocabolario Italiano (vocab_src) per leggere l'input
    tokens = [vocab_src.word2index.get(w, 3) for w in sentence_pulita.split(' ')]
    src_tensor = torch.tensor(tokens).unsqueeze(0).to(device) 

    with torch.no_grad():
        enc_output, dec_hidden = encoder(src_tensor)
        dec_input = torch.tensor([[1]], device=device) 
        translated_words = []

        for _ in range(max_length):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            predicted_id = predictions.argmax(1).item()

            if predicted_id == 2:
                break

            # Usiamo il vocabolario Bergamasco (vocab_trg) per scrivere l'output
            translated_word = vocab_trg.index2word.get(predicted_id, "<unk>")
            translated_words.append(translated_word)
            dec_input = torch.tensor([[predicted_id]], device=device)

    return " ".join(translated_words)

# ==========================================
# IL TEST FINALE CORRETTO!
# ==========================================
frase_test = "è veloce come un gatto al sole"

traduzione = translate_ita_to_berg(frase_test, encoder_model, decoder_model, vocab_ita, vocab_berg)

print("-" * 50)
print(f"ITALIANO: {frase_test}")
print(f"PREDETTO IN BERGAMASCO: {traduzione}")
print("-" * 50)

--------------------------------------------------
ITALIANO: è veloce come un gatto al sole
PREDETTO IN BERGAMASCO: l'è ü bradipo .
--------------------------------------------------
